# Seed-23 diagnostic — 5-seed comparison (Colab)

Targeted diagnostic to characterize why seed 23 is the recurring negative outlier (reports 026, 028, 029, 030, death-rerun). For each of seeds {17, 11, 23, 1, 2} captures:

- Codebook atom-pair geometry (sanity: same codebook → identical stats; if not, suggests floating-point determinism issue)
- Position-vector geometry per scale (substrate-seed dependent)
- Test set composition (target token frequency, entropy, top tokens)
- Baseline retrieval trace stats at W=2 (top_score distribution, fraction committed vs meta-stable, gate_signal distribution)
- Phase 4 candidate dynamics on 300 cues (n_stored, gate_signal distribution per scale)

**Prereqs:** codebook at `MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt`, A100 runtime, Run all.

**Expected output:** cell 7 prints a 5-seed comparison table. Look for *seed 23* differing from the positive seeds (17, 11) in: (a) higher `top_score_mean` (more committed retrievals) or (b) lower `frac_metastable` or (c) lower `gate_signal_mean` — any of these would explain why Phase 4 has less to discover for that seed.

In [ ]:
# 1. Clone the repo.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI

In [ ]:
# 2. Mount Drive and copy codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps.
!pip install -q torchhd datasets

In [ ]:
# 4. Sanity check.
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 5. Run the 5-seed diagnostic. ~5 min on A100.
import subprocess, os, time
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_path = 'reports/seed_diagnostic_colab.log'
os.makedirs('reports', exist_ok=True)
print(f'starting at {time.strftime("%H:%M:%S")}')
with open(log_path, 'w') as logf:
    proc = subprocess.run(
        ['python', 'experiments/37_seed_diagnostic.py',
         '--device', 'cuda',
         '--seeds', '17', '11', '23', '1', '2'],
        stdout=logf, stderr=subprocess.STDOUT,
    )
print(f'exit={proc.returncode}  finished at {time.strftime("%H:%M:%S")}')
!tail -50 {log_path}

In [ ]:
# 6. Copy results back to Drive.
import shutil, os
dst = '/content/drive/MyDrive/neuro-ai/results/seed_diagnostic'
if os.path.isdir('reports/seed_diagnostic'):
    shutil.copytree('reports/seed_diagnostic', dst, dirs_exist_ok=True)
shutil.copy('reports/seed_diagnostic_colab.log',
            '/content/drive/MyDrive/neuro-ai/results/seed_diagnostic_colab.log')
print('saved to', dst)
!ls {dst}

In [ ]:
# 7. Comparison table — the diagnostic output you paste back.
import json
from pathlib import Path

results = []
for s in [17, 11, 23, 1, 2]:
    p = Path(f'reports/seed_diagnostic/seed_{s}_diagnostic.json')
    if p.exists():
        results.append(json.loads(p.read_text()))

print(f'{"seed":>4} {"atom_cos_mean":>14} {"target_entropy":>14} {"top_score_mean":>15} {"top_score_std":>14}  {"frac_committed":>15} {"frac_metastable":>16} {"gate_sig_mean":>14} {"p4_stored":>10}')
for r in results:
    cA = r['condA_w2']
    print(f'{r["seed"]:>4} '
          f'{r["atom_pair_geometry"]["mean"]:>14.4f} '
          f'{r["test_set"]["target_entropy_bits"]:>14.3f} '
          f'{cA["top_score_mean"]:>15.4f} '
          f'{cA["top_score_std"]:>14.4f}  '
          f'{cA["frac_committed"]:>15.3f} '
          f'{cA["frac_metastable"]:>16.3f} '
          f'{cA["gate_signal_mean"]:>14.5f} '
          f'{r["phase4_300cues"]["n_stored"]:>10d}')

print()
# Per-scale candidate signal
print(f'{"seed":>4}  per-scale gate_signal mean (W=2 / W=3 / W=4):')
for r in results:
    ps = r['phase4_300cues']['per_scale_gate_signal']
    print(f'{r["seed"]:>4}   '
          f'W2 mean={ps["2"]["mean"]:.5f} above={ps["2"]["n_above_0p05"]:>3}/{ps["2"]["n_cues"]}   '
          f'W3 mean={ps["3"]["mean"]:.5f} above={ps["3"]["n_above_0p05"]:>3}/{ps["3"]["n_cues"]}   '
          f'W4 mean={ps["4"]["mean"]:.5f} above={ps["4"]["n_above_0p05"]:>3}/{ps["4"]["n_cues"]}')